# GraceDB BNS/NSBH SESN 3D Crossmatch

This notebook downloads public GraceDB production superevents passing the configured FAR threshold, keeps superevents passing the configured BNS/NSBH probability cut, downloads the best available 3D sky map, and crossmatches stripped-envelope supernovae from TNS.

The workflow is:

1. Download and filter the TNS public catalog for SESN-like types near the configured distance limit.
2. Query GraceDB superevents, fetch `p_astro.json` classifications, and download one best available multiorder FITS skymap per passing superevent.
3. Reproduce the temporal match from `how_many_SESN.ipynb` using the configured discovery-time window.
4. Run a configurable 3D credible-volume crossmatch twice: once with SN luminosity distances from the SHOES cosmology and once with `Planck18`.
5. If any SN lands inside the configured 3D credible volume, save diagnostic overlap plots to `gracedb_sesn_3d_plots/`.


In [21]:
from pathlib import Path
from io import BytesIO
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import requests

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM, Planck18
from astropy.time import Time

import healpy as hp
import matplotlib.pyplot as plt

from ligo.gracedb.rest import GraceDb
from ligo.skymap.io import read_sky_map
from ligo.skymap import distance, moc
from ligo.skymap.postprocess import contour, crossmatch, find_greedy_credible_levels
from ligo.skymap.postprocess.cosmology import dVC_dVL_for_DL


In [26]:
# TNS settings copied from the existing workflow.
TNS_API_KEY = '174094777967c4c1438cdfd6.00935564'
TNS_BOT_NAME = 'DESIRT_Bot'
TNS_BOT_ID = 105220
CATALOG_URL = 'https://www.wis-tns.org/system/files/tns_public_objects/tns_public_objects.csv.zip'
TNS_CSV_SKIPROWS = 1
SESN_TYPE_REGEX = 'Ib|Ic|IIb'

# GraceDB settings.
GRACEDB_SERVICE_URL = 'https://gracedb.ligo.org/api/'
GRACEDB_CATEGORY = 'Production'
GRACEDB_MAX_RESULTS = None
GRACEDB_QUERY_SIGFIGS = 12

# Event-selection settings.
FAR_THRESHOLD_PER_YEAR = 2.0
JULIAN_YEAR_DAYS = 365.25
SECONDS_PER_DAY = (1 * u.day).to_value(u.s)
JULIAN_YEAR_SECONDS = (JULIAN_YEAR_DAYS * u.day).to_value(u.s)
FAR_THRESHOLD_HZ = FAR_THRESHOLD_PER_YEAR / JULIAN_YEAR_SECONDS
GRACEDB_QUERY = (
    f'category: {GRACEDB_CATEGORY} '
    f'far < {FAR_THRESHOLD_HZ:.{GRACEDB_QUERY_SIGFIGS}g}'
)
MIN_BNS_NSBH_PROB_SUM = 0.9
DEFAULT_CLASSIFICATION_PROBABILITY = 0.0

# Temporal/spatial crossmatch settings.
TEMPORAL_WINDOW_DAYS = 14
MAX_SESN_DISTANCE_MPC = 500
CREDIBLE_LEVEL = 0.50
REQUIRE_2D_CREDIBLE_LEVEL = False

# crossmatch(..., cosmology=False) ranks the 3D posterior by probability
# density per luminosity-distance volume, matching the units in the skymaps.
# The two cosmology runs below differ by the redshift-to-luminosity-distance
# conversion used for each SN.
USE_COMOVING_VOLUME_RANKING = True

# SHOES/Planck cosmology settings.
SHOES_H0 = 73.04
SHOES_OM0 = 0.3
CMB_TEMPERATURE_K = 2.725
SHOES = FlatLambdaCDM(H0=SHOES_H0, Om0=SHOES_OM0, Tcmb0=CMB_TEMPERATURE_K * u.K)
COSMOLOGIES = {
    'SHOES': SHOES,
    'Planck18': Planck18,
}

# Local output directories.
SKYMAP_DIR = Path('gracedb_skymaps')
PLOT_DIR = Path('gracedb_sesn_3d_plots')
SKYMAP_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

# GraceDB skymap file-selection priorities. Lower is preferred.
SKYMAP_PRIORITY_BILBY_MULTIORDER = 0
SKYMAP_PRIORITY_BAYESTAR_MULTIORDER = 10
SKYMAP_PRIORITY_ANY_MULTIORDER = 20
SKYMAP_PRIORITY_BAYESTAR_FITS_GZ = 30
SKYMAP_PRIORITY_ANY_FITS_GZ = 40
SKYMAP_PRIORITY_ANY_FITS = 50
SKYMAP_VERSIONED_FILE_PRIORITY_PENALTY = 100
SKYMAP_PRIORITY_IGNORE = 1000

# Plot/raster settings.
PLOT_OUTPUT_FORMAT = 'pdf'
PLOT_HEALPIX_ORDER = 8
PLOT_FIGSIZE = (12, 8.5)
PLOT_MARGINS = (0.02, 0.04, 0.02, 0.12)
PLOT_PROBABILITY_MIN = 0.0
PLOT_BBOX_INCHES = 'tight'
PLOT_PERCENT_SCALE = 100

# Projected 3D contour grid settings.
DISTANCE_GRID_SIZE = 1000
DISTANCE_GRID_DISTMEAN_MULTIPLIER = 6.0
DISTANCE_GRID_SN_DISTANCE_MULTIPLIER = 1.05
DISTANCE_SHELL_VARIANCE_DENOMINATOR = 12.0
GAUSSIAN_EXPONENT_FACTOR = -0.5

# Contour and marker styling.
PLOT_2D_CONTOUR_COLOR = 'white'
PLOT_2D_CONTOUR_LINEWIDTH = 1.2
PLOT_2D_CONTOUR_ALPHA = 0.9
PLOT_3D_CONTOUR_COLOR = '#ffb000'
PLOT_3D_CONTOUR_LINEWIDTH = 1.8
PLOT_3D_CONTOUR_ALPHA = 0.95
PLOT_GRATICULE_COLOR = '0.7'
PLOT_GRATICULE_ALPHA = 0.4
SN_MARKER = '*'
SN_MARKER_SIZE = 150
SN_MARKER_COLOR = 'crimson'
SN_MARKER_EDGE_COLOR = 'white'
SN_MARKER_EDGE_WIDTH = 0.9
SN_MARKER_ZORDER = 10

# Plot legend box styling.
LEGEND_X = 0.035
LEGEND_Y = 0.035
LEGEND_FONT_SIZE = 8.5
LEGEND_HORIZONTAL_ALIGNMENT = 'left'
LEGEND_VERTICAL_ALIGNMENT = 'bottom'
LEGEND_BOX_STYLE = 'round'
LEGEND_BOX_PAD = 0.45
LEGEND_FACE_COLOR = 'white'
LEGEND_EDGE_COLOR = 'black'
LEGEND_LINEWIDTH = 0.8
LEGEND_ALPHA = 0.88


In [3]:
def download_tns_table():
    with requests.post(
        CATALOG_URL,
        headers={
            'user-agent': 'tns_marker{{"tns_id":"{id}","type":"bot","name":"{name}"}}'.format(
                id=TNS_BOT_ID,
                name=TNS_BOT_NAME,
            )
        },
        data={'api_key': (None, TNS_API_KEY)},
    ) as response:
        response.raise_for_status()
        return response.content


def clean_tns_catalog(df):
    keep_cols = [
        'name',
        'ra',
        'declination',
        'redshift',
        'type',
        'discoverydate',
        'reporting_group',
        'internal_names',
    ]
    df = df[keep_cols].copy()
    df = df[df['type'].str.contains(SESN_TYPE_REGEX, na=False, regex=True)].copy()
    df['discoverydate'] = pd.to_datetime(df['discoverydate'], errors='coerce', utc=True)
    df['redshift'] = pd.to_numeric(df['redshift'], errors='coerce')
    df['ra'] = pd.to_numeric(df['ra'], errors='coerce')
    df['declination'] = pd.to_numeric(df['declination'], errors='coerce')

    for label, cosmo in COSMOLOGIES.items():
        dist_col = f'dist_mpc_{label}'
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            df[dist_col] = cosmo.luminosity_distance(df['redshift'].to_numpy()).to_value(u.Mpc)

    required = ['discoverydate', 'redshift', 'ra', 'declination']
    df = df.dropna(subset=required)
    near = np.zeros(len(df), dtype=bool)
    for label in COSMOLOGIES:
        near |= df[f'dist_mpc_{label}'] < MAX_SESN_DISTANCE_MPC
    return df[near].reset_index(drop=True)


In [5]:
tns_data = download_tns_table()
tns_raw = pd.read_csv(BytesIO(tns_data), skiprows=TNS_CSV_SKIPROWS, compression='zip', low_memory=False)
df_sesn = clean_tns_catalog(tns_raw)

df_sesn

,name,ra,declination,redshift,type,discoverydate,reporting_group,internal_names,dist_mpc_SHOES,dist_mpc_Planck18
0,2026ntr,136.857168,29.126264,0.060000,SN Ic-BL,2026-05-28 06:16:05.088000+00:00,ATLAS,"ATLAS26gls, ZTF26aaywncn",257.478075,277.813361
1,2026gzf,149.928635,0.418438,0.034300,SN Ic-BL,2026-03-22 07:35:25.002000+00:00,ZTF,"ZTF26aaonmha, GOTO26ckg, ATLAS26dgs, PS18csa",144.480970,155.925025
2,2026ezd,139.682636,-20.148352,0.095200,SN Ic-BL,2026-02-26 05:34:49.002000+00:00,ZTF,"ZTF26aajrjtr, GOTO26cex, PS26bua",418.609558,451.540056
3,2026naz,203.083744,1.570124,0.030332,SN Ib,2026-05-13 06:09:59.002000+00:00,ZTF,"ZTF26aaxgyxg, , GOTO26fcn, PS26brx",127.392579,137.487617
4,2026oao,187.577012,47.209222,0.014585,SN Ic-BL,2026-06-02 04:05:56.999000+00:00,ZTF,"ZTF26aaywqbl, ATLAS26gqi",60.537201,65.342916
...,...,...,...,...,...,...,...,...,...,...
1391,2016Q,122.582750,19.446722,0.103000,SN Ibn,2016-01-07 09:15:50+00:00,Pan-STARRS,PS16hy,455.285325,491.069619
1392,2016P,209.379583,6.097500,0.014600,SN Ic-BL,2016-01-19 04:36:28+00:00,RANSSP,NaN,60.600150,65.410854
1393,2016M,109.157292,67.892306,0.036000,SN IIb,2016-01-15 15:47:00+00:00,NaN,NaN,151.831665,163.855648
1394,2016G,45.990417,43.401000,0.009000,SN Ic-BL,2016-01-09 22:26:24+00:00,NaN,", Gaia16acf",37.197287,40.152037


In [6]:
def as_float(value, default=np.nan):
    try:
        if value is None:
            return default
        return float(value)
    except (TypeError, ValueError):
        return default


def response_to_bytes(response):
    if hasattr(response, 'read'):
        data = response.read()
    else:
        data = response.content
    if isinstance(data, str):
        data = data.encode('utf-8')
    return data


def response_to_text(response):
    return response_to_bytes(response).decode('utf-8', errors='replace')


def unversioned_file_names(files):
    return [name for name in files if not re.search(r',\d+$', name)]


def choose_pastro_file(superevent, files):
    names = unversioned_file_names(files)
    by_lower = {name.lower(): name for name in names}
    pipeline = str(superevent.get('preferred_event_data', {}).get('pipeline', '')).strip()
    candidates = []
    if pipeline:
        candidates.extend([
            f'{pipeline}.p_astro.json',
            f'{pipeline.lower()}.p_astro.json',
            f'{pipeline.upper()}.p_astro.json',
        ])
    candidates.extend(['p_astro.json', 'pastro.json'])
    for candidate in candidates:
        if candidate.lower() in by_lower:
            return by_lower[candidate.lower()]

    p_astro_files = sorted(
        name for name in names
        if name.lower().endswith('.p_astro.json') or name.lower().endswith('p_astro.json')
    )
    return p_astro_files[0] if p_astro_files else None


def load_classification(client, superevent_id, pastro_file):
    if not pastro_file:
        return {}, None
    payload = response_to_text(client.files(superevent_id, pastro_file))
    data = json.loads(payload)
    return data, pastro_file


def skymap_priority(name):
    lower = name.lower()
    version_penalty = (
        SKYMAP_VERSIONED_FILE_PRIORITY_PENALTY
        if re.search(r',\d+$', lower)
        else 0
    )
    if not lower.endswith(('.multiorder.fits', '.fits.gz', '.fits')):
        return SKYMAP_PRIORITY_IGNORE + version_penalty
    if 'bilby.multiorder.fits' in lower:
        return SKYMAP_PRIORITY_BILBY_MULTIORDER + version_penalty
    if 'bayestar.multiorder.fits' in lower:
        return SKYMAP_PRIORITY_BAYESTAR_MULTIORDER + version_penalty
    if lower.endswith('.multiorder.fits'):
        return SKYMAP_PRIORITY_ANY_MULTIORDER + version_penalty
    if 'bayestar' in lower and lower.endswith('.fits.gz'):
        return SKYMAP_PRIORITY_BAYESTAR_FITS_GZ + version_penalty
    if lower.endswith('.fits.gz'):
        return SKYMAP_PRIORITY_ANY_FITS_GZ + version_penalty
    if lower.endswith('.fits'):
        return SKYMAP_PRIORITY_ANY_FITS + version_penalty
    return SKYMAP_PRIORITY_IGNORE + version_penalty


def choose_skymap_file(files):
    names = list(files)
    candidates = [name for name in names if skymap_priority(name) < SKYMAP_PRIORITY_IGNORE]
    if not candidates:
        return None
    return sorted(candidates, key=lambda name: (skymap_priority(name), name.lower()))[0]


def safe_file_part(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))


def download_gracedb_file(client, superevent_id, filename, outdir=SKYMAP_DIR):
    outdir.mkdir(exist_ok=True)
    local_name = f'{safe_file_part(superevent_id)}__{safe_file_part(filename)}'
    path = outdir / local_name
    if not path.exists():
        payload = response_to_bytes(client.files(superevent_id, filename))
        path.write_bytes(payload)
    return path


def gps_to_utc(gps_time):
    if pd.isna(gps_time):
        return pd.NaT
    return pd.Timestamp(Time(float(gps_time), format='gps').to_datetime(), tz='UTC')


In [7]:
def fetch_gracedb_bns_nsbh_superevents():
    client = GraceDb(service_url=GRACEDB_SERVICE_URL)
    rows = []

    for superevent in client.superevents(query=GRACEDB_QUERY, max_results=GRACEDB_MAX_RESULTS):
        sid = superevent.get('superevent_id')
        preferred = superevent.get('preferred_event_data', {}) or {}
        far_hz = as_float(superevent.get('far'), as_float(preferred.get('far')))
        far_per_year = far_hz * JULIAN_YEAR_SECONDS
        if not np.isfinite(far_per_year) or far_per_year >= FAR_THRESHOLD_PER_YEAR:
            continue

        try:
            files = client.files(sid).json()
        except Exception as exc:
            rows.append({
                'superevent_id': sid,
                'status': f'file_list_failed: {exc}',
                'far_hz': far_hz,
                'far_per_year': far_per_year,
            })
            continue

        pastro_file = choose_pastro_file(superevent, files)
        try:
            classification, classification_file = load_classification(client, sid, pastro_file)
        except Exception as exc:
            classification = {}
            classification_file = pastro_file
            status = f'p_astro_failed: {exc}'
        else:
            status = 'ok'

        p_bns = as_float(classification.get('BNS'), DEFAULT_CLASSIFICATION_PROBABILITY)
        p_nsbh = as_float(classification.get('NSBH'), DEFAULT_CLASSIFICATION_PROBABILITY)
        if not (p_bns + p_nsbh > MIN_BNS_NSBH_PROB_SUM):
            continue

        skymap_file = choose_skymap_file(files)
        skymap_path = None
        if skymap_file:
            try:
                skymap_path = download_gracedb_file(client, sid, skymap_file)
            except Exception as exc:
                status = f'skymap_download_failed: {exc}'

        gps_time = as_float(superevent.get('t_0'), as_float(preferred.get('gpstime')))
        gw_time = gps_to_utc(gps_time)
        rows.append({
            'superevent_id': sid,
            'gw_time': gw_time,
            'gps_time': gps_time,
            'far_hz': far_hz,
            'far_per_year': far_per_year,
            'p_bns': p_bns,
            'p_nsbh': p_nsbh,
            'p_bbh': as_float(classification.get('BBH'), DEFAULT_CLASSIFICATION_PROBABILITY),
            'p_terrestrial': as_float(classification.get('Terrestrial'), DEFAULT_CLASSIFICATION_PROBABILITY),
            'classification_file': classification_file,
            'preferred_event': preferred.get('graceid'),
            'pipeline': preferred.get('pipeline'),
            'search': preferred.get('search'),
            'instruments': preferred.get('instruments'),
            'labels': ','.join(superevent.get('labels', [])),
            'skymap_file': skymap_file,
            'skymap_path': str(skymap_path) if skymap_path else None,
            'status': status,
        })

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values('gw_time').reset_index(drop=True)


gracedb_events = fetch_gracedb_bns_nsbh_superevents()
gracedb_events

,superevent_id,gw_time,gps_time,far_hz,far_per_year,p_bns,p_nsbh,p_bbh,p_terrestrial,classification_file,preferred_event,pipeline,search,instruments,labels,skymap_file,skymap_path,status
0,S190425z,2019-04-25 08:18:42.011549+00:00,1.240216e+09,4.537648e-13,1.431973e-05,9.994026e-01,0.000000,0.000000e+00,5.974329e-04,p_astro.json,G330564,gstlal,AllSky,"L1,V1","ADVOK,SKYMAP_READY,EMBRIGHT_READY,PASTRO_READY...",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
1,S190814bv,2019-08-14 21:11:16.012957+00:00,1.249852e+09,2.032625e-33,6.414477e-26,0.000000e+00,0.997890,0.000000e+00,0.000000e+00,p_astro.json,G347305,gstlal,AllSky,"H1,L1,V1","PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_READY,PAS...",bayestar.multiorder.fits,gracedb_skymaps/S190814bv__bayestar.multiorder...,ok
2,S190822c,2019-08-22 01:30:36.589203+00:00,1.250473e+09,6.145185e-18,1.939273e-10,1.000000e+00,0.000000,0.000000e+00,5.223321e-10,p_astro.json,G347846,gstlal,AllSky,"L1,V1","ADVNO,SKYMAP_READY,EMBRIGHT_READY,PASTRO_READY...",bayestar.multiorder.fits,gracedb_skymaps/S190822c__bayestar.multiorder....,ok
3,S190910d,2019-09-10 01:26:56.242676+00:00,1.252114e+09,3.717180e-09,1.173053e-01,0.000000e+00,0.975899,0.000000e+00,2.410074e-02,p_astro.json,G350002,spiir,HighMass,"H1,L1","PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_READY,PAS...",bayestar.multiorder.fits,gracedb_skymaps/S190910d__bayestar.multiorder....,ok
4,S191117j,2019-11-17 06:08:59.454868+00:00,1.258006e+09,1.114482e-18,3.517037e-11,0.000000e+00,1.000000,0.000000e+00,1.080973e-10,p_astro.json,G354833,gstlal,AllSky,"H1,L1","ADVNO,EM_Selected,SKYMAP_READY,EMBRIGHT_READY,...",bayestar.multiorder.fits,gracedb_skymaps/S191117j__bayestar.multiorder....,ok
5,S191205ah,2019-12-05 21:52:45.568738+00:00,1.259618e+09,1.248393e-08,3.939628e-01,0.000000e+00,0.932102,0.000000e+00,6.789774e-02,p_astro.json,G356642,gstlal,AllSky,"H1,L1,V1","EM_READY,ADVOK,EM_Selected,SKYMAP_READY,EMBRIG...",bayestar.multiorder.fits,gracedb_skymaps/S191205ah__bayestar.multiorder...,ok
6,S191220af,2019-12-20 12:24:51.690032+00:00,1.260880e+09,3.963175e-10,1.250683e-02,9.963550e-01,0.000000,0.000000e+00,3.644999e-03,p_astro.json,G357916,gstlal,AllSky,"L1,V1","EM_READY,PE_READY,ADVNO,EM_Selected,SKYMAP_REA...",bayestar.multiorder.fits,gracedb_skymaps/S191220af__bayestar.multiorder...,ok
7,S200116ah,2020-01-16 11:57:19.170712+00:00,1.263211e+09,2.028966e-12,6.402930e-05,0.000000e+00,0.999934,0.000000e+00,6.579270e-05,p_astro.json,G360499,gstlal,AllSky,"H1,L1","EM_READY,PE_READY,ADVNO,EM_Selected,SKYMAP_REA...",bayestar.multiorder.fits,gracedb_skymaps/S200116ah__bayestar.multiorder...,ok
8,S230529ay,2023-05-29 18:15:37.746094+00:00,1.369419e+09,1.975121e-10,6.233008e-03,3.060151e-01,0.624028,0.000000e+00,6.995704e-02,pycbc.p_astro.json,G408702,pycbc,AllSky,L1,"EM_READY,PE_READY,ADVOK,SKYMAP_READY,EMBRIGHT_...",Bilby.multiorder.fits,gracedb_skymaps/S230529ay__Bilby.multiorder.fits,ok
9,S230715bw,2023-07-15 19:06:14.952637+00:00,1.373483e+09,7.843396e-09,2.475188e-01,0.000000e+00,0.906988,8.832113e-02,4.690587e-03,spiir.p_astro.json,G417598,spiir,AllSky,"H1,L1","EM_READY,PE_READY,ADVNO,SKYMAP_READY,EMBRIGHT_...",bayestar.multiorder.fits,gracedb_skymaps/S230715bw__bayestar.multiorder...,ok


In [17]:
def temporal_crossmatch_sesn_to_gw(df_sesn, gw_events):
    if df_sesn.empty or gw_events.empty:
        return pd.DataFrame()

    chunks = []
    for _, gw in gw_events.iterrows():
        if pd.isna(gw['gw_time']):
            continue
        start = gw['gw_time'] - pd.Timedelta(days=TEMPORAL_WINDOW_DAYS)
        end = gw['gw_time'] + pd.Timedelta(days=TEMPORAL_WINDOW_DAYS)
        sn = df_sesn[
            (df_sesn['discoverydate'] >= start) &
            (df_sesn['discoverydate'] <= end)
        ].copy()
        if sn.empty:
            continue
        sn['superevent_id'] = gw['superevent_id']
        sn['gw_time'] = gw['gw_time']
        sn['gps_time'] = gw['gps_time']
        sn['days_from_gw'] = (sn['discoverydate'] - gw['gw_time']).dt.total_seconds() / SECONDS_PER_DAY
        for col in [
            'far_per_year',
            'p_bns',
            'p_nsbh',
            'p_bbh',
            'p_terrestrial',
            'preferred_event',
            'pipeline',
            'search',
            'instruments',
            'skymap_file',
            'skymap_path',
            'status',
        ]:
            sn[f'gw_{col}'] = gw.get(col)
        chunks.append(sn)

    if not chunks:
        return pd.DataFrame()
    return pd.concat(chunks, ignore_index=True).sort_values(['gw_time', 'discoverydate']).reset_index(drop=True)


df_sesn_gracedb_temporal = temporal_crossmatch_sesn_to_gw(df_sesn, gracedb_events)
df_sesn_gracedb_temporal

,name,ra,declination,redshift,type,discoverydate,reporting_group,internal_names,dist_mpc_SHOES,dist_mpc_Planck18,...,gw_p_nsbh,gw_p_bbh,gw_p_terrestrial,gw_preferred_event,gw_pipeline,gw_search,gw_instruments,gw_skymap_file,gw_skymap_path,gw_status
0,2019dgw,287.799129,48.492651,0.095000,SN Ib,2019-04-11 11:47:45+00:00,ZTF,ZTF19aapwnmb,417.673722,450.531339,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
1,2019dps,314.673208,-54.185181,0.044107,SN Ic,2019-04-15 08:23:59+00:00,ASAS-SN,"ASASSN-19kn, Gaia19bky",187.128529,201.934218,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
2,2019dxr,227.863142,5.200552,0.040000,SN Ib,2019-04-15 09:06:36+00:00,ZTF,"ZTF19aarnqys, ATLAS19hxo, PS19wz",169.197122,182.590277,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
3,2019eev,149.274417,8.069490,0.042000,SN IIb,2019-04-21 03:42:38+00:00,ZTF,"ZTF19aarhhfx, ATLAS19jgk, PS19aoi",177.916433,191.996604,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
4,2019dwf,221.131631,70.455981,0.051000,SN IIb,2019-04-21 06:49:26+00:00,ZTF,"ZTF19aarfkch, PS19cxp",217.453479,234.645116,...,0.000000,0.0,0.000597,G330564,gstlal,AllSky,"L1,V1",GW190425_PublicationSamples.multiorder.fits,gracedb_skymaps/S190425z__GW190425_Publication...,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2025bdy,223.101063,3.716175,0.035700,SN Ib,2025-02-08 11:48:31.003000+00:00,ZTF,"ZTF25aafkxdu, BGEM J145224.28+034258.1, GOTO25...",150.533198,162.454754,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok
174,2025bcn,209.837753,-40.073401,0.013000,SN IIb,2025-02-09 02:08:38.112000+00:00,ATLAS,"ATLAS25bne, GOTO25afo, PS25awq",53.893524,58.172600,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok
175,2025bvu,135.096355,52.064095,0.031647,SN Ib,2025-02-14 09:39:11.808000+00:00,Pan-STARRS,"PS25un, ATLAS25brb, ZTF25aagowuq",133.044973,143.586361,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok
176,2025cav,138.033482,41.317581,0.092000,SN Ic,2025-02-18 22:57:23.328000+00:00,GOTO,"GOTO25amj, ATLAS25btd, PS25yk, WFST0499ckcnz",403.663710,435.429892,...,0.550467,0.0,0.076666,G552809,pycbc,AllSky,"H1,L1",Bilby.multiorder.fits,gracedb_skymaps/S250206dm__Bilby.multiorder.fits,ok


In [18]:
temporal_summary = (
    df_sesn_gracedb_temporal
    .groupby(['superevent_id', 'gw_time'], dropna=False)
    .size()
    .rename('n_temporal_sesn')
    .reset_index()
    if not df_sesn_gracedb_temporal.empty
    else pd.DataFrame(columns=['superevent_id', 'gw_time', 'n_temporal_sesn'])
)

if not gracedb_events.empty:
    gracedb_temporal_summary = gracedb_events.merge(
        temporal_summary, on=['superevent_id', 'gw_time'], how='left'
    )
    gracedb_temporal_summary['n_temporal_sesn'] = (
        gracedb_temporal_summary['n_temporal_sesn'].fillna(0).astype(int)
    )
else:
    gracedb_temporal_summary = pd.DataFrame()

display_cols = [
    'superevent_id',
    'gw_time',
    'far_per_year',
    'p_bns',
    'p_nsbh',
    'pipeline',
    'search',
    'skymap_file',
    'n_temporal_sesn',
    'status',
]
gracedb_temporal_summary[[c for c in display_cols if c in gracedb_temporal_summary.columns]]

,superevent_id,gw_time,far_per_year,p_bns,p_nsbh,pipeline,search,skymap_file,n_temporal_sesn,status
0,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,9.994026e-01,0.000000,gstlal,AllSky,GW190425_PublicationSamples.multiorder.fits,13,ok
1,S190814bv,2019-08-14 21:11:16.012957+00:00,6.414477e-26,0.000000e+00,0.997890,gstlal,AllSky,bayestar.multiorder.fits,20,ok
2,S190822c,2019-08-22 01:30:36.589203+00:00,1.939273e-10,1.000000e+00,0.000000,gstlal,AllSky,bayestar.multiorder.fits,20,ok
3,S190910d,2019-09-10 01:26:56.242676+00:00,1.173053e-01,0.000000e+00,0.975899,spiir,HighMass,bayestar.multiorder.fits,13,ok
4,S191117j,2019-11-17 06:08:59.454868+00:00,3.517037e-11,0.000000e+00,1.000000,gstlal,AllSky,bayestar.multiorder.fits,3,ok
5,S191205ah,2019-12-05 21:52:45.568738+00:00,3.939628e-01,0.000000e+00,0.932102,gstlal,AllSky,bayestar.multiorder.fits,13,ok
6,S191220af,2019-12-20 12:24:51.690032+00:00,1.250683e-02,9.963550e-01,0.000000,gstlal,AllSky,bayestar.multiorder.fits,16,ok
7,S200116ah,2020-01-16 11:57:19.170712+00:00,6.402930e-05,0.000000e+00,0.999934,gstlal,AllSky,bayestar.multiorder.fits,20,ok
8,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,3.060151e-01,0.624028,pycbc,AllSky,Bilby.multiorder.fits,10,ok
9,S230715bw,2023-07-15 19:06:14.952637+00:00,2.475188e-01,0.000000e+00,0.906988,spiir,AllSky,bayestar.multiorder.fits,7,ok


**About 2D vs 3D Ranking**

`ligo.skymap.postprocess.crossmatch` reports two different rankings. `searched_prob_2d` is the sky-only credible level after marginalizing over distance. `searched_prob_vol` is a 3D voxel credible level ranked by posterior density per volume at `(RA, Dec, distance)`. These are not nested quantities, so `searched_prob_vol` can be much smaller than `searched_prob_2d` when the SN distance falls on a high-density distance slice even if the sky position is not in the highest-probability sky pixels.


In [27]:
def add_crossmatch_columns(sn_rows, result, cosmology_label, distance_column):
    out = sn_rows.copy().reset_index(drop=True)
    out['cosmology'] = cosmology_label
    out['distance_column'] = distance_column
    out['sn_dist_mpc'] = out[distance_column]
    out['searched_area_deg2'] = np.atleast_1d(result.searched_area)
    out['searched_prob_2d'] = np.atleast_1d(result.searched_prob)
    out['offset_deg'] = np.atleast_1d(result.offset)
    out['searched_prob_dist'] = np.atleast_1d(result.searched_prob_dist)
    out['searched_vol_mpc3'] = np.atleast_1d(result.searched_vol)
    out['searched_prob_vol'] = np.atleast_1d(result.searched_prob_vol)
    out['searched_prob_3d_density_rank'] = out['searched_prob_vol']
    out['probdensity_vol'] = np.atleast_1d(result.probdensity_vol)
    out['credible_volume_mpc3'] = result.contour_vols[0] if result.contour_vols else np.nan
    out['credible_area_deg2'] = result.contour_areas[0] if result.contour_areas else np.nan
    out['inside_2d_credible_level'] = out['searched_prob_2d'] <= CREDIBLE_LEVEL
    out['inside_3d_credible_level'] = out['searched_prob_vol'] <= CREDIBLE_LEVEL
    return out


def failed_spatial_rows(sn_rows, status, cosmology=np.nan):
    out = sn_rows.copy()
    out['spatial_status'] = status
    out['cosmology'] = cosmology
    out['inside_2d_credible_level'] = False
    out['inside_3d_credible_level'] = False
    return out


def run_3d_spatial_crossmatch(temporal_matches, gw_events):
    if temporal_matches.empty or gw_events.empty:
        return pd.DataFrame()

    event_lookup = gw_events.set_index('superevent_id', drop=False)
    chunks = []
    skymap_cache = {}

    for superevent_id, sn_rows in temporal_matches.groupby('superevent_id'):
        if superevent_id not in event_lookup.index:
            continue
        event = event_lookup.loc[superevent_id]
        skymap_path = event.get('skymap_path')
        if not skymap_path or not Path(skymap_path).exists():
            chunks.append(failed_spatial_rows(sn_rows, 'missing_skymap'))
            continue

        if skymap_path not in skymap_cache:
            try:
                skymap_cache[skymap_path] = read_sky_map(skymap_path, moc=True)
            except Exception as exc:
                chunks.append(failed_spatial_rows(sn_rows, f'skymap_read_failed: {exc}'))
                continue
        skymap = skymap_cache[skymap_path]
        if 'DISTMU' not in skymap.colnames:
            chunks.append(failed_spatial_rows(sn_rows, 'skymap_has_no_distance_columns'))
            continue

        for cosmology_label in COSMOLOGIES:
            distance_column = f'dist_mpc_{cosmology_label}'
            valid = sn_rows[np.isfinite(sn_rows[distance_column])].copy()
            if valid.empty:
                continue
            coords = SkyCoord(
                ra=valid['ra'].to_numpy() * u.deg,
                dec=valid['declination'].to_numpy() * u.deg,
                distance=valid[distance_column].to_numpy() * u.Mpc,
                frame='icrs',
            )
            try:
                result = crossmatch(
                    skymap,
                    coords,
                    contours=(CREDIBLE_LEVEL,),
                    cosmology=USE_COMOVING_VOLUME_RANKING,
                )
                out = add_crossmatch_columns(valid, result, cosmology_label, distance_column)
                out['spatial_status'] = 'ok'
            except Exception as exc:
                out = failed_spatial_rows(valid, f'crossmatch_failed: {exc}', cosmology_label)
                out['distance_column'] = distance_column
            chunks.append(out)

    if not chunks:
        return pd.DataFrame()
    df = pd.concat(chunks, ignore_index=True)
    sort_cols = [c for c in ['superevent_id', 'name', 'cosmology'] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)
    return df.reset_index(drop=True)


df_sesn_gracedb_3d = run_3d_spatial_crossmatch(df_sesn_gracedb_temporal, gracedb_events)
df_sesn_gracedb_3d

,name,ra,declination,redshift,type,discoverydate,reporting_group,internal_names,dist_mpc_SHOES,dist_mpc_Planck18,...,searched_prob_dist,searched_vol_mpc3,searched_prob_vol,searched_prob_3d_density_rank,probdensity_vol,credible_volume_mpc3,credible_area_deg2,inside_2d_credible_level,inside_3d_credible_level,spatial_status
0,2019dgw,287.799129,48.492651,0.095000,SN Ib,2019-04-11 11:47:45+00:00,ZTF,ZTF19aapwnmb,417.673722,450.531339,...,1.000000,1.661173e+08,1.000004,1.000004,5.397701e-22,2.144799e+06,2400.292101,False,False,ok
1,2019dgw,287.799129,48.492651,0.095000,SN Ib,2019-04-11 11:47:45+00:00,ZTF,ZTF19aapwnmb,417.673722,450.531339,...,0.999999,1.367708e+08,1.000004,1.000004,1.117510e-19,2.144799e+06,2400.292101,False,False,ok
2,2019dps,314.673208,-54.185181,0.044107,SN Ic,2019-04-15 08:23:59+00:00,ASAS-SN,"ASASSN-19kn, Gaia19bky",187.128529,201.934218,...,0.830019,7.130358e+07,1.000004,1.000004,3.358170e-14,2.144799e+06,2400.292101,False,False,ok
3,2019dps,314.673208,-54.185181,0.044107,SN Ic,2019-04-15 08:23:59+00:00,ASAS-SN,"ASASSN-19kn, Gaia19bky",187.128529,201.934218,...,0.741566,5.741121e+07,1.000001,1.000001,5.835956e-13,2.144799e+06,2400.292101,False,False,ok
4,2019dwf,221.131631,70.455981,0.051000,SN IIb,2019-04-21 06:49:26+00:00,ZTF,"ZTF19aarfkch, PS19cxp",217.453479,234.645116,...,0.947668,4.535013e+07,0.999969,0.999969,7.454480e-12,2.144799e+06,2400.292101,False,False,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,2025bvu,135.096355,52.064095,0.031647,SN Ib,2025-02-14 09:39:11.808000+00:00,Pan-STARRS,"PS25un, ATLAS25brb, ZTF25aagowuq",133.044973,143.586361,...,0.022690,2.424860e+09,0.999996,0.999996,1.882578e-257,1.666301e+06,221.882883,False,False,ok
352,2025cav,138.033482,41.317581,0.092000,SN Ic,2025-02-18 22:57:23.328000+00:00,GOTO,"GOTO25amj, ATLAS25btd, PS25yk, WFST0499ckcnz",403.663710,435.429892,...,0.776453,1.106575e+10,0.999996,0.999996,0.000000e+00,1.666301e+06,221.882883,False,False,ok
353,2025cav,138.033482,41.317581,0.092000,SN Ic,2025-02-18 22:57:23.328000+00:00,GOTO,"GOTO25amj, ATLAS25btd, PS25yk, WFST0499ckcnz",403.663710,435.429892,...,0.691387,1.106575e+10,0.999996,0.999996,0.000000e+00,1.666301e+06,221.882883,False,False,ok
354,2025drb,176.154584,1.885731,0.032000,SN Ic,2025-02-19 17:11:02.400000+00:00,WFST,"WFST0521ofmq, PS25aku",134.564118,145.225446,...,0.031100,4.013170e+09,0.999996,0.999996,0.000000e+00,1.666301e+06,221.882883,False,False,ok


In [28]:
if df_sesn_gracedb_3d.empty or not {'spatial_status', 'inside_3d_credible_level'}.issubset(df_sesn_gracedb_3d.columns):
    coincidence_sne = pd.DataFrame()
else:
    coincidence_mask = (
        (df_sesn_gracedb_3d['spatial_status'] == 'ok')
        & (df_sesn_gracedb_3d['inside_3d_credible_level'] == True)
    )
    if REQUIRE_2D_CREDIBLE_LEVEL:
        coincidence_mask &= df_sesn_gracedb_3d['inside_2d_credible_level'] == True
    coincidence_sne = df_sesn_gracedb_3d[coincidence_mask].copy()

coincidence_display_cols = [
    'superevent_id',
    'gw_time',
    'gw_far_per_year',
    'gw_p_bns',
    'gw_p_nsbh',
    'name',
    'type',
    'discoverydate',
    'days_from_gw',
    'redshift',
    'cosmology',
    'sn_dist_mpc',
    'searched_prob_2d',
    'searched_prob_3d_density_rank',
    'searched_prob_dist',
    'searched_area_deg2',
    'credible_area_deg2',
    'credible_volume_mpc3',
    'inside_2d_credible_level',
    'inside_3d_credible_level',
    'ra',
    'declination',
    'reporting_group',
    'internal_names',
]
coincidence_sne[[c for c in coincidence_display_cols if c in coincidence_sne.columns]]

,superevent_id,gw_time,gw_far_per_year,gw_p_bns,gw_p_nsbh,name,type,discoverydate,days_from_gw,redshift,...,searched_prob_dist,searched_area_deg2,credible_area_deg2,credible_volume_mpc3,inside_2d_credible_level,inside_3d_credible_level,ra,declination,reporting_group,internal_names
10,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019ebq,SN Ib/c,2019-04-25 12:11:31+00:00,0.161678,0.037000,...,0.602471,255.564915,2400.292101,2.144799e+06,True,True,255.326411,-7.002923,Pan-STARRS,PS19qp
11,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019ebq,SN Ib/c,2019-04-25 12:11:31+00:00,0.161678,0.037000,...,0.499138,255.564915,2400.292101,2.144799e+06,True,True,255.326411,-7.002923,Pan-STARRS,PS19qp
14,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019eff,SN IIb,2019-04-25 13:59:31+00:00,0.236678,0.050000,...,0.936488,3.147351,2400.292101,2.144799e+06,True,True,248.413109,13.910201,Pan-STARRS,"PS19sh, ZTF19aasckkq"
15,S190425z,2019-04-25 08:18:42.011549+00:00,1.431973e-05,0.999403,0.000000,2019eff,SN IIb,2019-04-25 13:59:31+00:00,0.236678,0.050000,...,0.881450,3.147351,2400.292101,2.144799e+06,True,True,248.413109,13.910201,Pan-STARRS,"PS19sh, ZTF19aasckkq"
38,S190814bv,2019-08-14 21:11:16.012957+00:00,6.414477e-26,0.000000,0.997890,2019npv,SN Ib,2019-08-16 07:45:07+00:00,1.440173,0.056000,...,0.374119,4.802989,7.403113,2.628749e+04,True,True,13.384619,-23.832957,DECam-GROWTH,"DG19wxnjc, PS19eqi"
39,S190814bv,2019-08-14 21:11:16.012957+00:00,6.414477e-26,0.000000,0.997890,2019npv,SN Ib,2019-08-16 07:45:07+00:00,1.440173,0.056000,...,0.253252,4.802989,7.403113,2.628749e+04,True,True,13.384619,-23.832957,DECam-GROWTH,"DG19wxnjc, PS19eqi"
236,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023itz,SN Ib,2023-05-16 03:36:13.824000+00:00,-13.610694,0.022000,...,0.041734,25370.168307,8865.930556,1.676365e+07,False,True,327.389230,-15.220088,ATLAS,ATLAS23koz
237,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023itz,SN Ib,2023-05-16 03:36:13.824000+00:00,-13.610694,0.022000,...,0.029054,25370.168307,8865.930556,1.676365e+07,False,True,327.389230,-15.220088,ATLAS,ATLAS23koz
240,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023iwy,SN Ic-BL,2023-05-19 08:23:59+00:00,-10.410865,0.030000,...,0.161019,19209.752952,8865.930556,1.676365e+07,False,True,270.077591,26.408899,ZTF,"ZTF23aakmewi, ATLAS23kzz, PS23dmb"
241,S230529ay,2023-05-29 18:15:37.746094+00:00,6.233008e-03,0.306015,0.624028,2023iwy,SN Ic-BL,2023-05-19 08:23:59+00:00,-10.410865,0.030000,...,0.119537,19209.752952,8865.930556,1.676365e+07,False,True,270.077591,26.408899,ZTF,"ZTF23aakmewi, ATLAS23kzz, PS23dmb"


In [29]:
def raster_prob_from_moc(skymap, order=PLOT_HEALPIX_ORDER):
    raster = moc.rasterize(skymap, order=order)
    nside = hp.npix2nside(len(raster))
    if 'PROB' in raster.dtype.names:
        prob = np.asarray(raster['PROB'], dtype=float)
    else:
        prob = np.asarray(raster['PROBDENSITY'], dtype=float) * hp.nside2pixarea(nside)
    prob = np.nan_to_num(prob, nan=0.0, posinf=0.0, neginf=0.0)
    total = prob.sum()
    if total > 0:
        prob = prob / total
    return prob


def raster_3d_density_slice(skymap, dl_mpc, contour_level=CREDIBLE_LEVEL, order=PLOT_HEALPIX_ORDER):
    """Return an approximate projected 3D contour map at one distance."""
    raster = moc.rasterize(skymap, order=order)
    nside = hp.npix2nside(len(raster))
    dA = np.full(len(raster), hp.nside2pixarea(nside))

    dP_dA = np.asarray(raster['PROBDENSITY'], dtype=float)
    mu = np.asarray(raster['DISTMU'], dtype=float)
    sigma = np.asarray(raster['DISTSIGMA'], dtype=float)
    norm = np.asarray(raster['DISTNORM'], dtype=float)

    valid = (
        np.isfinite(dP_dA)
        & np.isfinite(mu)
        & np.isfinite(sigma)
        & np.isfinite(norm)
        & (dP_dA > 0)
        & (sigma > 0)
        & (norm > 0)
    )
    if not valid.any() or not np.isfinite(dl_mpc):
        return None, np.nan

    dP = np.zeros_like(dP_dA)
    dP[valid] = dP_dA[valid] * dA[valid]
    distmean, _ = distance.parameters_to_marginal_moments(
        dP[valid], mu[valid], sigma[valid]
    )

    max_r = max(
        DISTANCE_GRID_DISTMEAN_MULTIPLIER * distmean,
        DISTANCE_GRID_SN_DISTANCE_MULTIPLIER * dl_mpc,
    )
    if not np.isfinite(max_r) or max_r <= 0:
        return None, np.nan
    d_r = max_r / DISTANCE_GRID_SIZE
    r = d_r * np.arange(1, DISTANCE_GRID_SIZE)

    dV = (
        np.square(r) + np.square(d_r) / DISTANCE_SHELL_VARIANCE_DENOMINATOR
    ) * d_r * dA.reshape(-1, 1)
    radial_density = np.exp(
        GAUSSIAN_EXPONENT_FACTOR
        * np.square((r.reshape(1, -1) - mu.reshape(-1, 1)) / sigma.reshape(-1, 1))
    ) * (dP_dA * norm / (sigma * np.sqrt(2 * np.pi))).reshape(-1, 1)
    dP_grid = radial_density * dV
    dP_grid[~np.isfinite(dP_grid)] = 0

    rank_volume = dV
    if USE_COMOVING_VOLUME_RANKING:
        rank_volume = dV * dVC_dVL_for_DL(r).reshape(1, -1)
    density_grid = dP_grid / rank_volume
    density_grid[~np.isfinite(density_grid)] = 0

    order_idx = np.flipud(np.argsort(density_grid.ravel()))
    ranked_prob = dP_grid.ravel()[order_idx]
    ranked_density = density_grid.ravel()[order_idx]
    cumulative_prob = np.cumsum(ranked_prob)
    if cumulative_prob.size == 0 or cumulative_prob[-1] <= 0:
        return None, np.nan

    target_prob = contour_level * cumulative_prob[-1]
    threshold_idx = min(np.searchsorted(cumulative_prob, target_prob), len(ranked_density) - 1)
    density_threshold = ranked_density[threshold_idx]

    density_at_distance = np.exp(
        GAUSSIAN_EXPONENT_FACTOR * np.square((dl_mpc - mu) / sigma)
    ) * (dP_dA * norm / (sigma * np.sqrt(2 * np.pi)))
    if USE_COMOVING_VOLUME_RANKING:
        density_at_distance = density_at_distance / dVC_dVL_for_DL(dl_mpc)
    density_at_distance = np.nan_to_num(density_at_distance, nan=0.0, posinf=0.0, neginf=0.0)
    return density_at_distance, density_threshold


def draw_contours(prob, skymap, row):
    credible_2d = find_greedy_credible_levels(prob)
    for polygon in contour(credible_2d, [CREDIBLE_LEVEL], nest=True, degrees=True)[0]:
        polygon = np.asarray(polygon)
        if len(polygon):
            hp.projplot(
                polygon[:, 0],
                polygon[:, 1],
                lonlat=True,
                color=PLOT_2D_CONTOUR_COLOR,
                linewidth=PLOT_2D_CONTOUR_LINEWIDTH,
                alpha=PLOT_2D_CONTOUR_ALPHA,
            )

    density_slice, density_threshold = raster_3d_density_slice(
        skymap, row['sn_dist_mpc'], CREDIBLE_LEVEL
    )
    if density_slice is None or not np.isfinite(density_threshold) or density_threshold <= 0:
        return False

    drew_3d = False
    for polygon in contour(density_slice, [density_threshold], nest=True, degrees=True)[0]:
        polygon = np.asarray(polygon)
        if len(polygon):
            hp.projplot(
                polygon[:, 0],
                polygon[:, 1],
                lonlat=True,
                color=PLOT_3D_CONTOUR_COLOR,
                linewidth=PLOT_3D_CONTOUR_LINEWIDTH,
                alpha=PLOT_3D_CONTOUR_ALPHA,
            )
            drew_3d = True
    return drew_3d


def plot_3d_coincidence(row, gw_events, outdir=PLOT_DIR):
    outdir.mkdir(exist_ok=True)
    event = gw_events.set_index('superevent_id').loc[row['superevent_id']]
    skymap = read_sky_map(event['skymap_path'], moc=True)
    prob = raster_prob_from_moc(skymap)

    title = f"{row['superevent_id']} / {row['name']} ({row['type']}) / {row['cosmology']}"
    fig = plt.figure(figsize=PLOT_FIGSIZE)
    hp.mollview(
        prob,
        nest=True,
        fig=fig.number,
        title=title,
        unit='probability per pixel',
        cmap='viridis',
        min=PLOT_PROBABILITY_MIN,
        cbar=True,
        margins=PLOT_MARGINS,
    )
    hp.graticule(color=PLOT_GRATICULE_COLOR, alpha=PLOT_GRATICULE_ALPHA)

    drew_3d = False

    hp.projscatter(
        row['ra'],
        row['declination'],
        lonlat=True,
        marker=SN_MARKER,
        s=SN_MARKER_SIZE,
        color=SN_MARKER_COLOR,
        edgecolors=SN_MARKER_EDGE_COLOR,
        linewidths=SN_MARKER_EDGE_WIDTH,
        label=row['name'],
        zorder=SN_MARKER_ZORDER,
    )

    contour_note = (
        f'{PLOT_2D_CONTOUR_COLOR}: 2D {PLOT_PERCENT_SCALE * CREDIBLE_LEVEL:.0f}%; '
        f'{PLOT_3D_CONTOUR_COLOR}: projected 3D density-rank '
        f'{PLOT_PERCENT_SCALE * CREDIBLE_LEVEL:.0f}% at SN distance'
    )
    if not drew_3d:
        contour_note = (
            f'{PLOT_2D_CONTOUR_COLOR}: 2D {PLOT_PERCENT_SCALE * CREDIBLE_LEVEL:.0f}%; '
            f'{PLOT_3D_CONTOUR_COLOR}: projected 3D density-rank '
            f'{PLOT_PERCENT_SCALE * CREDIBLE_LEVEL:.0f}% unavailable for this slice'
        )

    legend_text = (
        f"delay = {row['days_from_gw']:+.2f} d\n"
        f"FAR = {row.get('gw_far_per_year', np.nan):.3g} yr^-1\n"
        f"p_NSNS = {row.get('gw_p_bns', np.nan):.3g}    p_NSBH = {row.get('gw_p_nsbh', np.nan):.3g}\n"
        f"D_L({row['cosmology']}) = {row['sn_dist_mpc']:.1f} Mpc\n"
        f"2D sky CL = {row['searched_prob_2d']:.3f}\n"
        f"3D density-rank = {row['searched_prob_3d_density_rank']:.3f}\n"
        f"distance CDF = {row['searched_prob_dist']:.3f}\n"
        f"credible area = {row['credible_area_deg2']:.1f} deg^2\n"
        f"credible 3D volume = {row['credible_volume_mpc3']:.3g} Mpc^3\n"
        f"{contour_note}"
    )
    fig.text(
        LEGEND_X,
        LEGEND_Y,
        legend_text,
        fontsize=LEGEND_FONT_SIZE,
        ha=LEGEND_HORIZONTAL_ALIGNMENT,
        va=LEGEND_VERTICAL_ALIGNMENT,
        bbox={
            'boxstyle': f'{LEGEND_BOX_STYLE},pad={LEGEND_BOX_PAD}',
            'facecolor': LEGEND_FACE_COLOR,
            'edgecolor': LEGEND_EDGE_COLOR,
            'linewidth': LEGEND_LINEWIDTH,
            'alpha': LEGEND_ALPHA,
        },
    )

    filename = (
        f"{safe_file_part(row['superevent_id'])}__"
        f"{safe_file_part(row['name'])}__"
        f"{safe_file_part(row['cosmology'])}.{PLOT_OUTPUT_FORMAT}"
    )
    path = outdir / filename
    fig.savefig(path, bbox_inches=PLOT_BBOX_INCHES)
    plt.close(fig)
    return path

In [25]:
plot_paths = []
if not coincidence_sne.empty:
    for _, row in coincidence_sne.iterrows():
        plot_paths.append(plot_3d_coincidence(row, gracedb_events))

pd.DataFrame({'plot_path': [str(path) for path in plot_paths]})

,plot_path
0,gracedb_sesn_3d_plots/S190425z__2019ebq__Planc...
1,gracedb_sesn_3d_plots/S190425z__2019ebq__SHOES...
2,gracedb_sesn_3d_plots/S190425z__2019eff__Planc...
3,gracedb_sesn_3d_plots/S190425z__2019eff__SHOES...
4,gracedb_sesn_3d_plots/S190814bv__2019npv__Plan...
5,gracedb_sesn_3d_plots/S190814bv__2019npv__SHOE...
6,gracedb_sesn_3d_plots/S230529ay__2023itz__Plan...
7,gracedb_sesn_3d_plots/S230529ay__2023itz__SHOE...
8,gracedb_sesn_3d_plots/S230529ay__2023iwy__Plan...
9,gracedb_sesn_3d_plots/S230529ay__2023iwy__SHOE...
